In [ ]:
print("Hello from GitHub → Colab!")

# basic example - accessing and visualizing data from google earth

* the notebook intends to give a basic example of how to work with google earth engine from which more elaborate things can be developed

## prerequisites

* google cloud account that will accessed below
* google cloud project with google earth eninge API enabled

In [ ]:
from google.colab import auth
import ee

import google
import pandas as pd
import numpy as np

import geemap

In [ ]:
# Authenticate the notebook.
auth.authenticate_user()

In [ ]:
# REPLACE WITH YOUR CLOUD PROJECT!
# PROJECT = 'your-project'
PROJECT = 'pa-gcp-projectmeid'

# Authenticate to Earth Engine.
credentials, _ = google.auth.default()
ee.Initialize(credentials, project=PROJECT, opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
!gcloud config set project {PROJECT}

## earth engine API offers an easy access way to satellite imagery

* the line ```ee.ImageCollection()```
  * LANDSAT/LC08/C02/T1 refers to asset ID for the Landsat 8 Operational Land Imager (OLI) & Thermal Infrared Sensor (TIRS) Collection 2, Tier 1 top-of-atmosphere data
  * ```.filterDate(start, end)``` returns images from the time range
  * to get rid of cloud cover the Landsat calibrations and SimpleLandsatCloudScore can be used via ```ee.Algorithms.Landsat.simpleComposite```


### Links:
* https://developers.google.com/earth-engine/apidocs/ee-imagecollection
* for visualisation: https://developers.google.com/earth-engine/apidocs/map-addlayer


In [ ]:
# The image input data is a 2018 cloud-masked median composite.
landsatCollection = ee.ImageCollection('LANDSAT/LC08/C02/T1').filterDate('2018-01-01', '2018-12-31')

composite = ee.Algorithms.Landsat.simpleComposite(
  collection=landsatCollection,
  asFloat=True
);

# Use geemap to visualize the imagery.
#Map = geemap.Map(center=(37.8, -122.5), zoom=12)
Map = geemap.Map(center=(22.3964, 114.1095), zoom=12)

Map.addLayer(
    composite,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3},
    'median composite',
    True)

In [ ]:
Map

## now the same procedure with sentinel 2 data

for sentinel data I haven't found something simlar like ee.Algorithms.Landsat.simpleComposite so one needs to do a bit more calculations. The Cloud Displacement Index (CDI) leverages the parallax shift between bands (due to clouds being elevated above the ground) to help distinguish clouds from intrinsically bright surfaces, improving on simple reflectance-based tests.

Rescaling is needed as well since sentinel bands are stored as UINT16 scaled by 10000.

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2') \
        .filterDate('2018-01-01', '2018-12-31')

s2CloudProb = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY') \
                  .filterDate('2018-01-01', '2018-12-31')

def mask_with_cdi(image):
    # Compute CDI
    cdi = ee.Algorithms.Sentinel2.CDI(image)

    # Get the matching cloud-probability image
    prob = s2CloudProb \
             .filter(ee.Filter.equals('system:index', image.get('system:index'))) \
             .first() \
             .select('probability')

    # Scale the cirrus band (QA60 not in L1C—use B10 for cirrus)
    cirrus = image.select('B10').multiply(0.0001)

    # Define cloud pixels:
    #  - high cloud-prob (>65%) *and* cdi < –0.5
    #  - OR strong cirrus signal (>0.01)
    isCloud = prob.gt(65).And(cdi.lt(-0.5)).Or(cirrus.gt(0.01))

    # Invert for mask (True = clear) and apply
    return image.updateMask(isCloud.Not())

# 2. Apply mask and build a 10 m median composite of the visible/NIR bands
clean = s2.map(mask_with_cdi)
composite = clean.select(['B2','B3','B4','B8']).median().toFloat()

Map = geemap.Map(center=(22.3764, 114.1095), zoom=12)

scaled = composite.divide(10000)
# 3. Display
Map.addLayer(
    scaled,
     {'bands':['B4','B3','B2'], 'min': 0, 'max': 0.3},
    'Cloud-free Composite (scaled 0–1)',
    True
    )

In [ ]:
Map